# v1.4 P4 · seed 1 재현 · 64M

이 노트북은 seed 1 하나를 fresh initialization에서 실행합니다. 같은 입력 번들로
seed 1과 seed 2 노트북을 각각 실행하세요. seed 1의 gate 실패도 seed 2를 생략할 이유가 아닙니다.
각 seed는 64,005,751 prediction tokens / 8,399 updates / cursor 537,536을 소비합니다.
모델·read4·학습률·데이터 순서·effective batch 64·microbatch 16은 seed 0과 동일합니다.

감사된 seed 0 동결 증빙을 확인한 뒤 현재 코드로 GPU smoke를 수행합니다. 지정 milestone의
select first macro CE로 checkpoint 하나를 선택하고 validation gate를 한 번 평가합니다.
실패 seed도 대체하지 않고 반환합니다. 두 seed가 끝난 뒤 반환 증빙을 로컬에서 감사하고
세 seed 전체의 결정을 동결해야 최종 test 노트북을 실행할 수 있습니다.
**이 노트북은 test 평가를 실행하지 않습니다. P4 완료가 아닌 재현 실행 전달물입니다.**

새 Colab CUDA 런타임에서 위→아래 실행합니다. 각 seed의 Drive 경로는 별개입니다.
중단 시 `RESUME=True`로 새 런타임에서 마지막 완전 index를 복구합니다.
완료 run도 `RESUME=True`로 회수할 수 있으며 학습·gate를 다시 실행하지 않습니다.
`gate_started.json`만 있고 gate 완료물이 없으면 반복 평가 없이 증빙을 반환하세요.


In [8]:
READY = True
assert READY, 'DRAFT: corrected P1 audit and CPU smoke are not yet packaged'
BUNDLE_NAME = 'v1_4_p4_bundle_r1.zip'
EXPECTED_SHA256 = '75a435a7da71fcd3cf954cc3c12cc7d0c66db693aacd37ffb4105fc97b092bb3'
RESUME = False  # 기존 Drive run을 이어갈 때만 True


## 1. 입력 업로드 및 checksum 검증

    제공된 ZIP을 업로드합니다. Drive에 미리 올렸다면 BUNDLE_PATH에 해당 절대 경로를 지정할 수
    있습니다. 전체 ZIP hash와 내부 모든 파일의 hash를 검증한 뒤 새 로컬 폴더에만 풉니다.


In [9]:
import hashlib, json, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive, files
drive.mount('/content/drive')
BUNDLE_PATH = '/content/drive/MyDrive/MI/inputs/' + BUNDLE_NAME
if BUNDLE_PATH is None:
    uploaded = files.upload()
    assert BUNDLE_NAME in uploaded, 'Upload the named bundle'
    BUNDLE_PATH = '/content/' + BUNDLE_NAME
    del uploaded
bundle = Path(BUNDLE_PATH)
def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
assert file_sha(bundle) == EXPECTED_SHA256, 'Wrong or incomplete input ZIP'
base = Path('/content/boolean_interp')
base.mkdir(exist_ok=True)
ROOT = Path(tempfile.mkdtemp(prefix='v1_4_', dir=base))
with zipfile.ZipFile(bundle) as archive:
    for entry in archive.infolist():
        path = ROOT / entry.filename
        assert path.resolve().is_relative_to(ROOT.resolve())
        assert (entry.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink not allowed'
    inventory = json.loads(archive.read('bundle_manifest.json'))
    assert len(archive.namelist()) == len(set(archive.namelist()))
    assert set(archive.namelist()) == set(inventory['files']) | {'bundle_manifest.json'}
    archive.extractall(ROOT)
for name, expected in inventory['files'].items():
    assert file_sha(ROOT / name) == expected, name
os.chdir(ROOT)
print('Verified input:', EXPECTED_SHA256, ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

## 2. seed 0의 기록된 환경을 기준으로 설치·검증

seed 0의 전체 lock에서 추출한 constraints와 주요 패키지의 정확한 버전을 사용합니다.
전체 원본 lock도 번들에 보존합니다. 같은 버전 설치가 불가능하면 오류를 그대로 반환해 주세요.
GPU/Python/시스템 차이는 smoke와 각 학습 세션의 environment ID에 기록됩니다.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'experiment_v1_4/p4/requirements-primary.lock.txt', '-c', 'experiment_v1_4/p4/requirements-seed0.constraints.txt', '--extra-index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
sys.path.insert(0, str(ROOT))
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
subprocess.run(['nvidia-smi'], check=True)
print('Disk:', shutil.disk_usage(ROOT))
print('RAM bytes:', os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES'))
from interp_v1_4.runtime import verify_inputs
print(verify_inputs(ROOT))
child_env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD='1')
subprocess.run([sys.executable, '-m', 'pytest', 'tests_v1_4', '-q'], check=True, env=child_env)


## 3. 필수 GPU smoke

    최대 길이 302, masking, read4 accumulation, bitwise resume, 모든 층 hook,
    SAE/TC k=4/16 각각 100 updates, probe, identity/full/sparse patch를 검사합니다.
    Debug checkpoint는 본학습 초기값으로 사용하지 않습니다.


In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
SMOKE = ROOT / 'gpu_smoke' / stamp
subprocess.run([sys.executable, '-m', 'interp_v1_4.smoke', '--root', str(ROOT),
                '--device', 'cuda', '--output', str(SMOKE)], check=True)
smoke = json.loads((SMOKE / 'smoke.json').read_text())
assert smoke['status'] == 'passed' and smoke['scope'] == 'cuda'
MICRO = smoke['cells']['deepwide12_read4']['selected_microbatch']
print('Frozen microbatch:', MICRO, 'effective batch: 64')
assert MICRO == 16, 'P4 must preserve seed 0 microbatch; use a suitable GPU'


## 4. 영속 저장 및 fresh/resume 준비

    같은 run 이름으로 새 학습을 덮어쓰지 않습니다. Drive에 run이 이미 있으면 RESUME=True로
    처음부터 새 런타임에서 진행합니다. 이전 GPU와 microbatch가 맞지 않으면 임의 변경 없이 중단합니다.


In [ ]:
from interp_v1_4.persistence import recover, read_index
PERSIST = Path('/content/drive/MyDrive/MI/v1_4/p4_r1/deepwide12_read4_seed1')
OUTPUT = ROOT / 'runs' / ('seed1_' + stamp)
resume_args = []
if RESUME:
    checkpoint = recover(PERSIST, OUTPUT)
    payload = torch.load(checkpoint, map_location='cpu', weights_only=False)
    assert payload['microbatch'] == MICRO, 'GPU smoke/checkpoint microbatch mismatch'
    # Finalized runs can be recovered for export without another training/gate call.
    resume_args = ['--resume', str(checkpoint)]
    del payload
else:
    assert not PERSIST.exists(), 'Persistent run exists: use RESUME, never overwrite'


## 5. seed 1 학습

    최초 50 updates 처리량과 VRAM을 기록합니다. init, milestone 및 15분 경과 update의 완전
    checkpoint를 Drive에 저장합니다. 중단/OOM 시 batch·예산을 줄이지 말고 index에서 재개합니다.
    학습 완료 뒤 gate 결과가 failed여도 증빙 ZIP을 회수합니다. test는 채점하지 않습니다.


In [ ]:
command = [sys.executable, '-u', '-m', 'interp_v1_4.cli', '--root', str(ROOT),
           '--stage', 'p4', '--cell', 'deepwide12_read4', '--lm-seed', '1', '--device', 'cuda',
           '--microbatch', str(MICRO), '--smoke-report', str(SMOKE / 'smoke.json'),
           '--output', str(OUTPUT), '--persistent-dir', str(PERSIST)] + resume_args
if not (OUTPUT / 'result.json').exists():
    subprocess.run(command, check=True)
result = json.loads((OUTPUT / 'result.json').read_text())
print({key: result[key] for key in ('status', 'actual_prediction_tokens', 'overshoot')})


## 6. 완전 저장 증빙 검증·다운로드 (중단 시에도 실행)

    Drive의 마지막 완전 index만 export합니다. 완료 run은 gate를 재채점하지 않고 recorded gate,
    checkpoint, cursor, hash를 검증합니다. 중단 run은 paused로 회수하며 완료로 표시하지 않습니다.
    ZIP과 checksum은 Drive에도 보관됩니다. 반환 파일을 로컬 감사에 전달해 주세요.


In [ ]:
export_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
EXPORT = ROOT / ('evidence_' + export_stamp)
EXPORTED_RUN = EXPORT / 'deepwide12_read4_seed1'
recover(PERSIST, EXPORTED_RUN)
shutil.copytree(SMOKE, EXPORT / 'gpu_smoke')
shutil.copytree(ROOT / 'experiment_v1_4/p4', EXPORT / 'delivery_contract')
(EXPORT / 'input_bundle.json').write_text(json.dumps({'name': BUNDLE_NAME, 'sha256': EXPECTED_SHA256}))
if (EXPORTED_RUN / 'result.json').exists():
    subprocess.run([sys.executable, 'scripts/verify_v1_4_evidence.py', str(EXPORTED_RUN),
                    '--output', str(EXPORT / 'verification.json')], check=True)
else:
    (EXPORT / 'resume_required.json').write_text(json.dumps({'status': 'paused', 'index': read_index(EXPORTED_RUN)}))
checksums = {str(p.relative_to(EXPORT)): file_sha(p) for p in sorted(EXPORT.rglob('*')) if p.is_file()}
(EXPORT / 'checksums.json').write_text(json.dumps(checksums, indent=2))
zip_path = Path(shutil.make_archive(str(ROOT / ('v1_4_seed1_evidence_' + export_stamp)), 'zip', EXPORT))
checksum = file_sha(zip_path)
checksum_path = zip_path.with_suffix('.zip.sha256')
checksum_path.write_text(checksum + '  ' + zip_path.name + '\n')
DELIVERY = PERSIST.parent / 'exports'
DELIVERY.mkdir(exist_ok=True)
shutil.copy2(zip_path, DELIVERY / zip_path.name)
assert file_sha(DELIVERY / zip_path.name) == checksum
shutil.copy2(checksum_path, DELIVERY / checksum_path.name)
files.download(str(checksum_path))
files.download(str(zip_path))
print('Drive evidence:', DELIVERY / zip_path.name)
